# 📗 동적 계획과 병렬 조사, 핸드오프

교안 01의 슈퍼바이저는 워커를 한 번에 하나씩 불렀습니다. 이 교안에서는 서로 기다릴 필요가 없는 조사를 **동시에** 돌리고, 조사 메모를 합쳐 쓴 글을 검수 루프로 다듬습니다. 예시는 **사내 출장 준비 팀**입니다. 출장 요청을 받으면 항공·숙소·규정·현지 이동을 나눠 조사하고, 메모를 합쳐 출장자 안내문을 쓰고, 검수자가 조사 메모와 대조합니다.

**학습 목표**

1. **`Send`로 작업 수만큼 같은 노드를 병렬 실행합니다.** 작업 수는 실행 중에 정합니다.
2. **병렬 결과를 합치는 리듀서를 고릅니다.** 이어 붙이기와 작업 ID별 교체를 구분합니다.
3. **계획 → 병렬 조사 → 종합 → 핸드오프 → 검수 루프**를 하나의 팀 그래프로 연결합니다.

## 1. 정해진 병렬 vs 계획된 병렬

`add_edge`로 미리 연결한 **정해진 병렬**은 요청이 무엇이든 같은 수의 노드를 실행합니다. **계획된 병렬**은 플래너가 요청을 읽고 필요한 작업 수를 정한 뒤 그만큼 나눠 맡깁니다.

- 부산 당일 출장, "항공편만 알려 주세요" → 작업 1개(부산 항공편)
- 도쿄와 오사카 중 출장지를 고르는 중, "두 도시의 항공편과 숙소를 비교해 주세요" → 작업 4개(도쿄 항공편, 도쿄 숙소, 오사카 항공편, 오사카 숙소)

네 작업은 서로의 결과를 기다리지 않으므로 동시에 조사할 수 있습니다. 결과를 맞춰 두 도시를 비교하는 일은 조사가 모두 끝난 뒤에 합니다.

<img src="images/07_fixed_vs_planned.png" width="1050" alt="정해진 병렬은 어떤 요청이든 항공·숙소·규정 3개 작업을 조사원 3명에게 맡기고, 계획된 병렬은 planner가 요청을 읽고 부산 당일 항공편 요청은 작업 1개, 도쿄·오사카 비교 요청은 도시마다 항공편과 숙소를 나눈 작업 4개로 정합니다.">

| | 정해진 병렬 | 계획된 병렬 |
|---|---|---|
| 연결 방법 | `add_edge` 여러 개 | 라우팅 함수가 `Send` 목록 반환 |
| 맞는 경우 | 출장 요청마다 같은 항목을 조사할 때 | 출장 요청마다 조사할 항목이 다를 때 (예: "항공편만" 요청 vs "두 도시 항공편·숙소 비교" 요청) |

## 2. 실습 준비

### 2-1. 라이브러리·모델·파일 경로 설정

`.env`에 `OPENAI_API_KEY`를 설정하고 새 커널에서 시작합니다. 모델과 경로 설정은 53일차와 같은 방식입니다.

In [ ]:
# 라이브러리: 환경 변수, 타입 힌트, LangChain 모델·프롬프트, 구조화 출력 스키마
import os
import json
import operator
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown

# material_dir는 output이 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# API 키: 실행 폴더에서 위로 올라가며 찾은 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: 환경 변수에 모델 이름이 없으면 기본 모델을 씁니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)

### 2-2. 사내 출장 자료와 조사 도구

항공편·숙소·출장 규정·현지 이동 정보를 작은 딕셔너리로 둡니다. 조사 도구 네 개는 각각 자료 하나만 읽습니다. 출발지는 서울로 고정합니다.

In [ ]:
# 항공편 데이터: 도시별 서울 출발 항공편(편명·출발·도착·요금). 금액 단위는 원입니다.
FLIGHTS = {
    "부산": [{"편명": "KE1403", "출발": "김포 07:30", "도착": "김해 08:35", "요금": 98000},
             {"편명": "OZ8105", "출발": "김포 09:00", "도착": "김해 10:05", "요금": 89000}],
    "제주": [{"편명": "7C101", "출발": "김포 07:00", "도착": "제주 08:10", "요금": 62000}],
    "도쿄": [{"편명": "KE2101", "출발": "김포 08:20", "도착": "하네다 10:30", "요금": 420000},
             {"편명": "OZ1085", "출발": "김포 09:00", "도착": "하네다 11:10", "요금": 385000},
             {"편명": "7C1102", "출발": "인천 07:40", "도착": "나리타 10:05", "요금": 245000}],
    "오사카": [{"편명": "LJ211", "출발": "인천 08:10", "도착": "간사이 09:55", "요금": 198000}],
}

# 숙소 데이터: 도시별 출장 숙소(이름·1박 요금·위치)
HOTELS = {
    "부산": [{"이름": "해운대 비즈호텔", "1박": 115000, "위치": "벡스코 도보 10분"},
             {"이름": "센텀 스테이", "1박": 138000, "위치": "벡스코 도보 5분"}],
    "제주": [{"이름": "제주시티 호텔", "1박": 98000, "위치": "제주공항 차로 10분"}],
    "도쿄": [{"이름": "시나가와 프라임", "1박": 210000, "위치": "도쿄 빅사이트 전철 25분"},
             {"이름": "아리아케 베이 호텔", "1박": 168000, "위치": "도쿄 빅사이트 도보 7분"}],
    "오사카": [{"이름": "난바 인", "1박": 129000, "위치": "인텍스 오사카 지하철 30분"}],
}

# 출장 규정 데이터: 주제(숙박·항공·일비·승인)마다 사내 규정 조항 하나씩
POLICY = {
    "숙박": "1박 상한은 국내 12만 원, 해외 20만 원입니다. 초과분은 개인 부담입니다.",
    "항공": "이코노미석만 이용합니다. 국제선은 출발 14일 전까지 예약합니다.",
    "일비": "하루 일비는 국내 3만 원, 해외 6만 원입니다. 현지 교통비를 포함합니다.",
    "승인": "해외 출장은 출발 7일 전까지, 국내 출장은 3일 전까지 팀장 승인을 받습니다.",
}

# 현지 이동 데이터: 도시별 공항에서 행사장까지 가는 방법
LOCAL = {
    "부산": "김해공항에서 벡스코까지 경전철과 지하철로 약 50분입니다.",
    "제주": "제주공항에서 시내까지 버스로 약 15분입니다.",
    "도쿄": "하네다공항에서 도쿄 빅사이트까지 리무진버스로 약 30분, 나리타공항에서는 약 90분입니다. 교통카드는 Suica를 씁니다.",
    "오사카": "간사이공항에서 인텍스 오사카까지 전철로 약 70분입니다.",
}

In [ ]:
from langchain_core.tools import tool

# 인자 타입을 Literal로 두면 고를 수 있는 도시 목록이 도구 인자 스키마에 담겨 모델에 전달됩니다.
City = Literal["부산", "제주", "도쿄", "오사카"]


# [도구] 항공편 조사: FLIGHTS만 읽습니다.
@tool
def search_flights(city: City) -> str:
    """서울에서 그 도시로 가는 항공편(편명, 출발, 도착, 이코노미 요금)을 반환합니다."""
    lines = []
    for flight in FLIGHTS[city]:
        # 예약 담당자가 읽는 형식 그대로 한 편에 한 줄씩 씁니다.
        lines.append(
            f"{flight['편명']}: {flight['출발']} 출발, {flight['도착']} 도착, 이코노미 {flight['요금']:,}원"
        )

    return "\n".join(lines)


# [도구] 숙소 조사: HOTELS만 읽습니다.
@tool
def search_hotels(city: City) -> str:
    """그 도시의 출장 숙소(이름, 1박 요금, 위치)를 반환합니다."""
    lines = []
    for hotel in HOTELS[city]:
        lines.append(f"{hotel['이름']}: 1박 {hotel['1박']:,}원, {hotel['위치']}")

    return "\n".join(lines)


# [도구] 출장 규정 조회: 도시가 아니라 규정 주제를 인자로 받습니다.
@tool
def lookup_policy(topic: Literal["숙박", "항공", "일비", "승인"]) -> str:
    """사내 출장 규정에서 한 주제의 조항을 반환합니다."""
    return POLICY[topic]


# [도구] 현지 이동 조회: LOCAL만 읽습니다.
@tool
def local_info(city: City) -> str:
    """그 도시의 공항에서 행사장까지 이동 방법을 반환합니다."""
    return LOCAL[city]

**실행 확인:** 도구는 `invoke`로 직접 불러 볼 수 있습니다.

In [ ]:
# 도구에는 인자 이름을 키로 한 딕셔너리를 넘깁니다.
print("도쿄 출장 숙소 목록:")
print(search_hotels.invoke({"city": "도쿄"}))

## 3. Send: 작업 수만큼 같은 노드 실행하기

### 3-1. Send의 모양

`Send("노드 이름", 넘길 값)` 하나가 작업 하나입니다. 조건부 엣지(`add_conditional_edges`)에 넣는 **라우팅 함수**가 `Send` 목록을 반환하면 LangGraph가 목록 길이만큼 같은 노드를 동시에 실행합니다.

요청이 "다음 주 부산·도쿄 출장 숙소 최저가 알려 줘"일 때:

1. 라우팅 함수가 `[Send("check_hotel", {"city": "부산"}), Send("check_hotel", {"city": "도쿄"})]`를 반환합니다.
2. LangGraph가 목록 길이(2)만큼 `check_hotel`을 **동시에** 실행합니다. 각 실행은 State 전체가 아니라 `{"city": ...}`만 받습니다.
3. 둘 다 끝나면 다음 노드 `collect`가 **한 번** 실행됩니다.

<img src="images/08_send_fanout.png" width="1050" alt="부산·도쿄 숙소 최저가 요청에서 라우팅 함수가 city 부산과 city 도쿄를 담은 Send 두 개를 보내고, check_hotel 두 개가 동시에 실행된 뒤 collect가 한 번 실행됩니다.">

LLM 없이 요청 문장에서 도시를 찾고, 도시마다 최저가 숙소를 확인하는 그래프로 연습합니다. 결과를 모으는 `notes` 필드의 `operator.add`(리스트를 이어 붙이는 합치기 규칙)는 4절에서 자세히 다룹니다.

<img src="images/node_flow/l2_hotel_overview.png" width="1050" alt="plan_checks가 도시마다 Send를 보내 check_hotel을 도시 수만큼 동시에 실행하고, 모두 끝나면 collect가 한 번 실행됩니다.">

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# [State] 숙소 확인 그래프의 공유 데이터. 주석 앞 이름은 그 필드에 값을 쓰는 노드입니다.
class HotelCheckState(TypedDict):
    request: str  # 출장자가 보낸 요청 문장
    cities: list[str]  # plan_checks: 조사할 도시
    # check_hotel: operator.add(기존, 새 값)는 '기존 + 새 값'이라 병렬 결과가 리스트로 이어 붙습니다.
    notes: Annotated[list[str], operator.add]
    summary: str  # collect: notes를 한 줄씩 묶은 요약

### 3-2. 도시 찾기 노드 (`plan_checks`)

요청 문장에 들어 있는 도시를 숙소 자료의 도시 이름과 하나씩 대조해 찾습니다. 이어지는 라우팅 함수 `assign_checks`는 찾은 도시마다 `Send`를 하나씩 만들므로 **목록 길이가 곧 병렬 작업 수**입니다.

<img src="images/node_flow/l2_hotel_plan_checks.png" width="1050" alt="plan_checks: 요청 문장에서 도시를 찾습니다. 이어서 라우팅 함수가 도시마다 Send를 보냅니다.">

In [ ]:
# [노드] 요청 문장에서 도시 찾기
def plan_checks(state: HotelCheckState):
    """요청 문장에 나온 도시를 숙소 자료의 도시 순서대로 찾아 반환합니다."""
    cities = []
    for city in HOTELS:
        # 요청 문장에 도시 이름이 들어 있으면 조사 대상에 넣습니다.
        if city in state["request"]:
            cities.append(city)

    return {"cities": cities}


# [라우팅 함수] 찾은 도시마다 Send 만들기
def assign_checks(state: HotelCheckState):
    """도시마다 Send 하나를 만들어 반환합니다."""
    # Send에는 check_hotel이 쓸 도시 하나만 담습니다. 전체 State는 넘어가지 않습니다.
    return [Send("check_hotel", {"city": city}) for city in state["cities"]]

**실행 확인:** 요청 문장에서 찾은 도시와 `assign_checks`가 만드는 `Send` 목록을 봅니다.

In [ ]:
# 노드와 라우팅 함수는 그래프 없이도 State 모양의 딕셔너리로 직접 호출할 수 있습니다.
check_state = {"request": "다음 주 부산·도쿄 출장 숙소 최저가 알려 줘"}
check_state.update(plan_checks(check_state))

print("찾은 도시:", check_state["cities"])

# 찾은 도시마다 Send가 하나씩 만들어집니다.
sends = assign_checks(check_state)
print("Send 목록:", sends)

# Send 하나가 check_hotel에 넘기는 값입니다. State 전체(request, cities)가 아니라 도시 하나뿐입니다.
print("check_hotel이 받는 값:", sends[0].arg)

### 3-3. 숙소 확인 노드 (`check_hotel`)

`Send` 하나마다 따로 실행되어 도시 하나의 최저가 숙소를 `notes`에 남깁니다.

`check_hotel`도 `add_node`로 등록하는 보통 노드입니다. 다른 점은 들어오는 입력 하나입니다.

| 실행되는 방식 | 노드가 받는 입력 |
|---|---|
| 일반 엣지(`add_edge`)로 실행 | 그래프 State 전체 |
| `Send("check_hotel", {"city": "부산"})`로 실행 | `Send`에 담은 값 `{"city": "부산"}`만 |

그래서 인자 이름을 `state`가 아니라 `payload`로 두었습니다. 반환값은 보통 노드와 같습니다. `{"notes": [...]}`를 돌려주면 리듀서(`operator.add`)가 기존 `notes` 뒤에 이어 붙입니다.

<img src="images/node_flow/l2_hotel_check_hotel.png" width="1050" alt="check_hotel: Send로 받은 도시 하나를 조사합니다. 도시 수만큼 동시에 실행됩니다.">

In [ ]:
# [보조 함수] 최저가 숙소 찾기: 도시에서 1박 요금이 가장 낮은 숙소를 돌려줍니다.
def cheapest_hotel(city):
    """그 도시에서 1박 요금이 가장 낮은 숙소를 반환합니다."""
    cheapest = HOTELS[city][0]
    for hotel in HOTELS[city]:
        # 지금까지 찾은 숙소보다 싸면 바꿉니다.
        if hotel["1박"] < cheapest["1박"]:
            cheapest = hotel

    return cheapest


# [노드] 도시 하나의 최저가 숙소 확인
def check_hotel(payload):
    """Send로 받은 도시 하나의 최저가 숙소를 notes에 남깁니다."""
    cheapest = cheapest_hotel(payload["city"])
    note = f"{payload['city']} 최저가: {cheapest['이름']} 1박 {cheapest['1박']:,}원 ({cheapest['위치']})"

    # 리스트로 반환해야 operator.add 리듀서가 다른 도시 결과와 이어 붙입니다.
    return {"notes": [note]}

### 3-4. 결과 모으기 노드 (`collect`)

모든 `check_hotel`이 끝난 뒤 **한 번만** 실행됩니다. 이때 `notes`에는 병렬 결과가 모두 모여 있습니다.

<img src="images/node_flow/l2_hotel_collect.png" width="1050" alt="collect: 모든 check_hotel이 끝난 뒤 한 번 실행되어 결과를 묶습니다.">

In [ ]:
# [노드] 병렬 결과 모으기
def collect(state: HotelCheckState):
    """notes에 모인 도시별 메모를 한 줄씩 이어 summary로 반환합니다."""
    return {"summary": "\n".join(state["notes"])}

### 3-5. 그래프 연결과 실행

`add_conditional_edges`에 `Send` 목록을 반환하는 함수를 넣습니다. 세 번째 인자 `["check_hotel"]`은 라우팅 함수가 `Send`로 보낼 수 있는 노드 목록입니다.

In [ ]:
# 그래프 연결: plan_checks -> check_hotel(도시 수만큼) -> collect
builder = StateGraph(HotelCheckState)
builder.add_node("plan_checks", plan_checks)
builder.add_node("check_hotel", check_hotel)
builder.add_node("collect", collect)

builder.add_edge(START, "plan_checks")
# Send 목록의 길이만큼 check_hotel이 동시에 실행됩니다.
builder.add_conditional_edges("plan_checks", assign_checks, ["check_hotel"])
# 병렬로 실행된 check_hotel이 모두 끝나야 collect로 넘어갑니다.
builder.add_edge("check_hotel", "collect")
builder.add_edge("collect", END)

hotel_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(hotel_app.get_graph().draw_mermaid_png()))

같은 그래프에 도시가 하나인 요청과 셋인 요청을 넣어 작업 수가 달라지는지 확인합니다.

In [ ]:
# 그래프는 그대로 두고 입력만 바꿉니다.
one_city = hotel_app.invoke({"request": "다음 주 부산 출장 숙소 최저가 알려 줘"})
three_cities = hotel_app.invoke({"request": "다음 달 도쿄·오사카·제주 순회 출장, 도시별 숙소 최저가 알려 줘"})

print("작업 수 부산 출장:", len(one_city["notes"]), "| 순회 출장:", len(three_cities["notes"]))

# collect가 모든 check_hotel의 메모를 모아 만든 요약입니다.
print("도시별 숙소 최저가:")
print(three_cities["summary"])

### 3-6. 노드의 `Command`로 병렬 실행하기

병렬 실행은 라우팅 함수에서만 되는 것이 아닙니다. 교안 01의 `Command(goto, update)`에서 `goto`에 노드를 여러 개 넣으면, 그 노드들이 다음 단계에서 동시에 실행됩니다. 넣는 방법은 두 가지입니다.

```python
# 1) 노드 이름 목록: 서로 다른 노드를 동시에 실행
return Command(goto=["노드1", "노드2"], update={"cities": cities})

# 2) Send 목록: 같은 노드를 입력만 바꿔 여러 번 실행
return Command(goto=[Send("check_hotel", {"city": "부산"}), Send("check_hotel", {"city": "도쿄"})],
               update={"cities": cities})
```

| | 노드 이름 목록 | `Send` 목록 |
|---|---|---|
| 노드가 받는 입력 | `update`가 반영된 State 전체 | `Send`에 담은 값만. State의 다른 필드는 받지 못함 |
| 같은 노드를 여러 번 | 안 됨. `["노드1", "노드1"]`이어도 한 번만 실행 | 됨. 입력을 달리해 몇 번이든 실행 |
| 실행 개수 | 코드에 적은 노드 수로 고정 | 실행 중에 정한 작업 수만큼 |

- `Command`는 노드가 돌려주는 값입니다. 라우팅 함수는 `Command`를 돌려줄 수 없고, 노드 이름이나 `Send`(또는 그 목록)만 돌려줍니다.
- 어느 방법이든 동시에 끝난 노드가 State의 같은 필드에 쓰면 리듀서가 필요합니다(4절).

**이 교안이 `Send`를 라우팅 함수에서 보내는 이유: 역할 나누기**

- 노드(`plan_checks`)는 일을 하고 State를 갱신하는 데만 씁니다.
- 라우팅 함수(`assign_checks`)는 그 결과를 보고 다음 노드를 연결해 실행하는 데만 씁니다.
- 그래서 둘을 따로 호출해 확인할 수 있고(3-2 실행 확인), `add_conditional_edges`만 봐도 어디로 갈 수 있는지 보입니다.
- 판단하면서 State도 함께 바꿔야 할 때는 노드의 `Command`를 씁니다. 6절의 `supervisor`가 그 예입니다.

## 4. 리듀서: 병렬 결과를 합치는 규칙

**리듀서**는 State의 한 필드(예: `notes`)에 들어온 새 값을 기존 값과 합치는 규칙입니다. 50일차에 대화를 누적한 `add_messages`도 리듀서입니다. 리듀서가 없는 필드는 새 값으로 통째로 바뀌고, 한 단계(동시에 실행되는 노드 묶음)에서 값을 하나만 받을 수 있습니다. 그래서 병렬 노드가 함께 쓰는 필드에는 리듀서가 꼭 필요합니다.

"부산·도쿄 출장 숙소 확인해 줘"를 처리하면 `check_hotel` 두 개가 동시에 `notes` 필드에 씁니다.

- 리듀서 없음: 어느 값을 남길지 정할 수 없어 `InvalidUpdateError`로 멈춥니다.
- `operator.add`: `["부산 최저가: …", "도쿄 최저가: …"]`처럼 이어 붙입니다(4-1).
- dict 병합: `{"부산": …, "도쿄": …}`처럼 키별로 저장합니다(4-2).

<img src="images/09_reducer.png" width="1050" alt="부산 최저가와 도쿄 최저가 메모가 같은 필드에 동시에 올 때, 리듀서가 없으면 InvalidUpdateError, operator.add는 한 줄로 이어 붙이기, dict 병합은 부산·도쿄 키별로 저장합니다.">

### 4-1. `operator.add`: 이어 붙이기

3절의 `notes`는 `Annotated[list[str], operator.add]`입니다. 두 부분으로 읽습니다.

- `operator`는 파이썬 표준 라이브러리 모듈로, `+` 같은 연산자를 함수로 쓸 수 있게 해 줍니다. `operator.add(a, b)`는 `a + b`와 같습니다. 리스트에 쓰면 `operator.add(["부산 최저가"], ["도쿄 최저가"])`가 `["부산 최저가", "도쿄 최저가"]`가 되어 두 리스트가 이어 붙습니다.
- `Annotated[list[str], operator.add]`는 "이 필드의 값은 `list[str]`이고, 새 값이 오면 `operator.add`로 합친다"는 뜻입니다. LangGraph는 노드가 `{"notes": [...]}`를 반환할 때마다 `operator.add(기존 값, 새 값)`을 불러 필드를 갱신합니다.

이어 붙이기는 **다시 조사하면 옛 결과도 그대로 남습니다.** `notes`에 "도쿄 최저가: 지난주 조사 결과"가 있을 때 도쿄를 다시 조사하면 도쿄 메모가 두 개가 됩니다.

In [ ]:
# 지난주 도쿄 메모를 넣고 시작합니다. 시작 입력의 notes도 리듀서를 거칩니다.
rerun = hotel_app.invoke({"request": "도쿄·오사카 출장 숙소 다시 확인해 줘", "notes": ["도쿄 최저가: 지난주 조사 결과"]})
print("notes 필드:", rerun["notes"])

**리듀서 필드에는 새 결과만 반환합니다.** 교안 01의 `run_worker`는 `state["results"] + [새 보고]`를 반환했습니다. 그 `results`는 리듀서가 없는 필드라 반환한 리스트가 필드를 통째로 바꾸기 때문입니다. 같은 습관으로 `operator.add` 필드에 기존 값까지 담아 반환하면, 리듀서가 기존 값 뒤에 기존 값을 한 번 더 붙입니다. 50일차의 노드도 `add_messages` 필드에는 새 메시지만 반환했습니다.

In [ ]:
# 리듀서가 쓰는 operator.add를 직접 불러 봅니다. notes 필드에는 이미 부산 메모가 있습니다.
old_notes = ["부산 최저가: 해운대 비즈호텔"]

# 노드가 새 결과만 반환하면 도쿄 메모 하나만 붙습니다.
print("새 결과만 반환:", operator.add(old_notes, ["도쿄 최저가: 아리아케 베이 호텔"]))

# 기존 값까지 담아 반환하면 부산 메모가 두 번 들어갑니다.
print("기존 값까지 반환:", operator.add(old_notes, old_notes + ["도쿄 최저가: 아리아케 베이 호텔"]))

### 4-2. dict 병합 리듀서: 작업 ID별 교체

리듀서는 `(기존 값, 새 값)`을 받는 평범한 함수로 직접 만들 수 있습니다. 결과를 **작업 ID(여기서는 도시 이름)를 키로 하는 dict**에 모으면 다시 조사한 작업만 바뀝니다.

- 기존: `{"도쿄": "지난주 결과"}` + 새 값: `{"도쿄": "오늘 결과", "오사카": "오늘 결과"}`
- 합친 값: `{"도쿄": "오늘 결과", "오사카": "오늘 결과"}`

5절부터 이 리듀서를 씁니다.

In [ ]:
# [리듀서] 키(작업 ID)별로 교체하는 dict 병합
def merge_results(old: dict, new: dict) -> dict:
    """기존 결과에 새 결과를 합칩니다. 같은 키는 새 값으로 바꿉니다."""
    # 기존 값을 복사한 뒤 새 결과를 넣습니다. 같은 키는 새 값으로 바뀝니다.
    merged = dict(old)
    merged.update(new)

    return merged


# 리듀서 함수를 직접 불러 기존 dict와 새 dict를 합쳐 봅니다.
print("병합 결과:", merge_results({"도쿄": "지난주", "부산": "지난주"}, {"도쿄": "오늘"}))

3절 그래프에서 `collect`를 빼고, 결과를 `merge_results` 리듀서를 붙인 `results` 필드에 도시 이름 키로 모읍니다.

<img src="images/node_flow/l2_merge_overview.png" width="1050" alt="plan_checks가 Send로 check_hotel을 도시 수만큼 실행합니다. 이 check_hotel은 도시 이름을 키로 results 필드에 씁니다.">

In [ ]:
# [State] results 필드에 merge_results 리듀서를 붙인 공유 데이터
class MergeState(TypedDict):
    request: str
    cities: list[str]
    results: Annotated[dict[str, str], merge_results]  # 도시 이름을 키로 저장, 같은 키는 새 값으로 교체

**도시 이름을 키로 쓰는 노드 (`check_hotel_by_city`)**: 그래프에는 `check_hotel`이라는 이름으로 등록합니다. 결과를 `{도시: 메모}` 모양으로 반환합니다. 키가 같으면 리듀서가 새 메모로 바꿉니다.

<img src="images/node_flow/l2_merge_check_hotel.png" width="1050" alt="check_hotel: 도시 이름을 키로 results에 씁니다. 같은 도시는 새 결과로 바뀝니다.">

In [ ]:
# [노드] 도시 이름을 키로 결과 쓰기
def check_hotel_by_city(payload):
    """최저가 숙소를 도시 이름 키로 results에 씁니다."""
    # cheapest_hotel로 그 도시의 최저가 숙소를 찾습니다.
    cheapest = cheapest_hotel(payload["city"])

    # 도시 이름이 키라서 같은 도시를 다시 조사하면 그 키의 값만 바뀝니다.
    return {"results": {payload["city"]: f"{cheapest['이름']} 1박 {cheapest['1박']:,}원"}}

In [ ]:
# plan_checks -> check_hotel 연결은 같고, collect 없이 check_hotel 다음에 끝납니다.
builder = StateGraph(MergeState)
builder.add_node("plan_checks", plan_checks)
builder.add_node("check_hotel", check_hotel_by_city)

builder.add_edge(START, "plan_checks")
builder.add_conditional_edges("plan_checks", assign_checks, ["check_hotel"])
builder.add_edge("check_hotel", END)

merge_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(merge_app.get_graph().draw_mermaid_png()))

In [ ]:
# 지난주 도쿄 결과를 넣고 도쿄·오사카를 다시 조사합니다. 시작 입력의 results도 리듀서를 거칩니다.
merged = merge_app.invoke({"request": "도쿄·오사카 출장 숙소 다시 확인해 줘", "results": {"도쿄": "지난주 조사 결과"}})
print("results 필드:", merged["results"])

## 5. 계획 → 병렬 조사 → 종합

### 5-1. 전체 흐름

3절의 `plan_checks`는 정해진 도시 이름만 찾았습니다. 이번에는 **LLM 플래너**가 요청을 읽고 작업 목록을 만듭니다. 작업마다 조사원 에이전트를 새로 만들어 **그 작업의 도구 하나만** 주고, 조사가 모두 끝나면 `synthesizer`가 메모를 합칩니다.

<img src="images/node_flow/l2_research_overview.png" width="1050" alt="planner가 작업 수만큼 Send로 researcher를 동시에 실행하고, 모두 끝나면 synthesizer가 한 번 실행됩니다.">

**종합**은 따로 조사한 메모를 맞춰 보고 하나의 기획 메모(`brief`, 브리프)로 합치는 단계입니다. 조사원은 자기 도구 결과만 보므로 메모끼리의 충돌은 여기서만 찾을 수 있습니다. 4절의 리듀서는 메모를 한 필드에 모을 뿐 내용을 비교하지 않습니다. 비교는 `synthesizer`의 몫입니다.

- 숙소 메모: 시나가와 프라임 1박 210,000원
- 규정 메모: 해외 숙박 1박 상한 200,000원
- synthesizer: "시나가와 프라임(1박 210,000원)은 해외 숙박 상한을 넘음"을 주의할 점으로 적음

<img src="images/10_synthesize.png" width="1050" alt="synthesizer가 숙소 메모(시나가와 프라임 1박 210,000원)와 규정 메모(해외 숙박 상한 1박 200,000원)를 맞춰 보고, 기획 메모를 안내문에 쓸 사실(아리아케 베이 호텔 1박 168,000원), 주의할 점(시나가와 프라임 상한 초과), 조사하지 못한 항목(공항에서 행사장 이동)으로 나눕니다.">

### 5-2. 상태 정의

5~6절은 같은 State를 씁니다. `findings`는 4-2의 `merge_results` 리듀서로 작업 ID별로 모읍니다.

In [ ]:
# [State] 출장 준비 팀 그래프의 공유 데이터
class TripState(TypedDict):
    # 처음 설정 (아래 주석 앞 이름은 그 필드에 값을 쓰는 곳입니다)
    request: str  # 출장 요청 원문
    max_rounds: int  # supervisor가 돌려보낼 수 있는 최대 횟수(다시 쓰기·다시 조사 합산)

    # planner·researcher·synthesizer
    tasks: list[dict]  # planner: 이번에 할 조사 작업
    findings: Annotated[dict[str, str], merge_results]  # researcher: 작업 ID별 조사 메모
    brief: dict  # synthesizer: 안내문에 쓸 사실·주의할 점·조사하지 못한 항목

    # writer·reviewer·supervisor
    draft: str  # writer: 안내문. request_review 도구의 draft 인자가 여기에 저장됩니다
    review: dict  # reviewer·request_research: 통과 여부·문제 종류·의견
    feedback: str  # supervisor: 다음 라운드에 전할 검수 의견 (planner는 반영 후 비움)
    round: int  # supervisor: 지금까지 돌려보낸 횟수
    status: str  # supervisor: 끝난 이유(진행 중, 통과, 라운드 상한 도달)

In [ ]:
# [보조 함수] 그래프 시작 입력 만들기: 출장 요청 하나로 모든 필드의 초깃값을 채웁니다.
def make_input(request, max_rounds=2):
    """출장 요청 하나로 시작 상태를 만듭니다. 결과 필드는 비우고 라운드는 0부터 셉니다."""
    # 그래프 실행과 노드 직접 호출이 모두 이 시작 상태를 씁니다.
    return {
        "request": request,
        "max_rounds": max_rounds,
        "tasks": [],
        "findings": {},
        "brief": {},
        "draft": "",
        "review": {},
        "feedback": "",
        "round": 0,
        "status": "진행 중",
    }

### 5-3. 조사 계획 노드 (`planner`)

출장 요청을 읽고 조사 작업 목록(`ResearchPlan`)을 만듭니다. 작업 ID는 LLM이 아니라 코드가 `자료원_도시`(예: `hotel_tokyo`)로 만듭니다. 그래서 같은 조사를 다시 하면 늘 같은 ID가 나오고, `merge_results`가 그 키의 메모만 새것으로 바꿉니다. 검수 의견(`feedback`)이 있으면 빠진 조사만 작업으로 만들고, 반영한 의견은 비웁니다(6절).

<img src="images/node_flow/l2_research_planner.png" width="1050" alt="planner: 출장 요청을 읽고 조사 작업 목록을 만듭니다. 작업마다 Send가 하나씩 나갑니다.">

In [ ]:
# 작업 하나는 자료원 하나, 도시 하나입니다. 작업 수는 tasks 목록의 길이로 정해집니다.
class Task(BaseModel):
    source: Literal["flight", "hotel", "policy", "local"] = Field(description="이 작업이 쓰는 자료원 하나")
    target: Literal["busan", "jeju", "tokyo", "osaka"] = Field(description="조사할 출장 도시의 영문 이름")
    query: str = Field(description="조사원에게 줄 한국어 지시 한 문장")

# 작업마다 조사원 LLM 호출이 동시에 실행되므로 필드 설명과 프롬프트에 작업 수 상한(5개)을 적습니다.
class ResearchPlan(BaseModel):
    tasks: list[Task] = Field(description="필요한 조사 작업 1~5개")

# 요청에 필요한 조사만 나누게 하고, 검수 의견이 있으면 빠진 조사만 다시 계획하게 합니다.
planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "출장 요청을 읽고 안내문에 필요한 조사만 작업으로 나누세요. 작업 하나는 자료원 하나만 씁니다. 출발지는 서울입니다. "
     "flight=항공편, hotel=숙소, policy=출장 규정(숙박·항공·일비·승인), local=공항에서 행사장 가는 길. "
     "요청에 없는 도시나 항목은 만들지 마세요. 자료원과 도시 조합마다 작업은 하나입니다. 작업은 1~5개입니다. "
     "검수 의견이 '없음'이 아니면 요청 전체가 아니라 그 의견에 적힌 빈 칸을 메우는 작업만 만드세요."),
    ("human", "출장 요청: {request}\n검수 의견: {feedback}"),
])

# 플래너 체인: 프롬프트 -> LLM -> ResearchPlan 구조화 출력
planner_chain = planner_prompt | llm.with_structured_output(ResearchPlan)

In [ ]:
# [노드] 조사 계획
def planner(state: TripState):
    """출장 요청과 검수 의견을 읽고 이번에 할 조사 작업 목록을 반환합니다."""
    feedback = state["feedback"]
    # 첫 계획에는 검수 의견이 없으므로 "없음"을 넘깁니다.
    if not feedback:
        feedback = "없음"

    # LLM이 요청과 검수 의견을 읽고 작업 목록을 만듭니다.
    plan = planner_chain.invoke({"request": state["request"], "feedback": feedback})

    tasks = []
    for task in plan.tasks:
        # Pydantic 객체를 dict로 바꿔 State와 Send에 그대로 담을 수 있게 합니다.
        task_dict = task.model_dump()
        # 작업 ID는 코드가 만듭니다. 같은 자료원·도시를 다시 조사하면 같은 ID가 나옵니다.
        task_dict["task_id"] = task_dict["source"] + "_" + task_dict["target"]
        tasks.append(task_dict)

    # 검수 의견은 이번 계획에 반영했으므로 비웁니다. writer가 이미 반영된 의견을 다시 읽지 않게 합니다.
    return {"tasks": tasks, "feedback": ""}


# [라우팅 함수] 작업마다 Send 만들기
def assign_research(state: TripState):
    """작업마다 Send 하나로 researcher에게 작업 정보만 넘깁니다."""
    return [Send("researcher", {"task": task}) for task in state["tasks"]]

**실행 확인:** 간단한 요청과 복잡한 요청에서 작업 수가 달라지는지 봅니다. LLM이 정하므로 실행마다 수가 조금 다를 수 있습니다.

In [ ]:
# 같은 planner에 간단한 요청과 복잡한 요청을 넣습니다.
busan_request = "다음 주 화요일 부산 벡스코에서 고객사 미팅이 있어요. 당일치기라 항공편만 알려 주세요."
tokyo_request = ("10월 21일부터 24일까지 도쿄 빅사이트 박람회에 3박 4일로 갑니다. "
                 "항공편, 숙소, 공항에서 행사장 가는 길, 출장 규정상 주의할 점을 정리해 주세요.")
busan_tasks = planner(make_input(busan_request))["tasks"]

# 도쿄 요청의 계획 결과는 node_state에 넣어 두고, 이어지는 researcher 확인에서 씁니다.
node_state = make_input(tokyo_request)
node_state.update(planner(node_state))

print("작업 수 부산 당일 출장:", len(busan_tasks), "| 도쿄 박람회 출장:", len(node_state["tasks"]))

# 작업 ID·자료원·지시문을 한 줄씩 봅니다.
for task in node_state["tasks"]:
    print(task["task_id"], task["source"], task["query"])

### 5-4. 조사 노드 (`researcher`)

`Send`가 넘긴 `{"task": ...}`만 받아, 작업의 `source`에 맞는 도구 하나만 가진 조사원을 `create_agent`로 그 자리에서 만듭니다(예: 항공 조사원은 숙소 도구를 볼 수 없음). 결과는 `{task_id: 메모}` 모양으로 반환하고, `merge_results` 리듀서가 병렬 결과를 모읍니다.

조사원은 다른 작업의 결과를 기다리지 않습니다. 예를 들어 공항에서 행사장 가는 길은 항공편이 어느 공항에 내리는지 모르는 채로 하네다·나리타 경로를 모두 조사하고, 어느 경로를 쓸지는 5-5의 `synthesizer`가 항공편 메모와 맞춰 봅니다.

<img src="images/node_flow/l2_research_researcher.png" width="1050" alt="researcher: 작업 하나를 그 작업의 도구 하나만 가진 에이전트로 조사합니다. 작업 수만큼 동시에 실행됩니다.">

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 자료원별로 허용할 도구입니다. 조사원은 자기 작업의 도구 하나만 받습니다.
TOOLS_BY_SOURCE = {
    "flight": [search_flights],
    "hotel": [search_hotels],
    "policy": [lookup_policy],
    "local": [local_info],
}

# 조사원 시스템 프롬프트: 도구를 먼저 부르고 도구 결과에 있는 내용만 정리하게 합니다.
researcher_system = ("출장 조사원입니다. 먼저 주어진 도구를 호출하고, 도구 결과에 나온 항목(편명, 숙소, 공항별 이동 방법 등)을 빠뜨리지 말고 짧게 정리하세요. 출발지는 서울입니다. "
                     "금액·시간·이름은 도구 결과 그대로 쓰고, 도구 결과에 없는 내용은 쓰지 마세요.")


# [노드] 작업 하나를 맡은 조사원
def researcher(payload):
    """작업의 도구 하나만 가진 에이전트로 조사하고 작업 ID를 키로 메모를 반환합니다."""
    task = payload["task"]

    # 작업마다 도구가 다르므로 에이전트를 이 자리에서 만듭니다.
    agent = create_agent(llm, tools=TOOLS_BY_SOURCE[task["source"]],
                         system_prompt=researcher_system)
    result = agent.invoke({"messages": [HumanMessage(task["query"])]})

    # 마지막 AI 메시지가 조사 정리입니다. 메시지 기록 전체는 State에 넣지 않습니다.
    return {"findings": {task["task_id"]: result["messages"][-1].text}}

**실행 확인:** 첫 작업 하나만 조사합니다.

In [ ]:
# Send가 넘기는 것과 같은 모양 {"task": ...}으로 직접 호출합니다.
first_task = node_state["tasks"][0]
first_result = researcher({"task": first_task})

print(first_task["task_id"], "조사 메모:")
print(first_result["findings"][first_task["task_id"]])

### 5-5. 조사 종합 노드 (`synthesizer`)

모든 조사 메모를 맞춰 보고 기획 메모(`Brief`)를 **안내문에 쓸 사실**, 규정과 어긋나거나 메모끼리 맞지 않는 **주의할 점**, 요청에 비해 **조사하지 못한 항목**으로 나눠 만듭니다. 먼저 조사 메모를 프롬프트에 넣을 문자열로 바꾸는 `format_findings`를 둡니다.

<img src="images/node_flow/l2_research_synthesizer.png" width="1050" alt="synthesizer: 모든 researcher가 끝난 뒤 한 번 실행되어 조사 메모를 합칩니다.">

In [ ]:
# [보조 함수] LLM 컨텍스트 만들기: researcher가 모은 조사 결과(findings)를 synthesizer·reviewer 프롬프트에 넣을 문자열로 바꿉니다.
def format_findings(findings):
    """작업 ID와 조사 메모를 한 문자열로 묶습니다."""
    blocks = []
    for task_id, text in findings.items():
        # 작업 ID를 대괄호로 붙여 어느 조사에서 나온 메모인지 모델이 구분하게 합니다.
        blocks.append(f"[{task_id}]\n{text}")

    return "\n\n".join(blocks)

In [ ]:
# 기획 메모 스키마: 안내문에 쓸 사실·주의할 점·조사하지 못한 항목
class Brief(BaseModel):
    facts: list[str] = Field(description="안내문에 쓸 사실. 금액·시간·이름은 메모 그대로")
    conflicts: list[str] = Field(description="출장 규정과 어긋나거나 메모끼리 맞지 않는 점. 없으면 빈 목록")
    gaps: list[str] = Field(description="요청에 비해 아직 조사하지 못한 것. 없으면 빈 목록")

# 메모끼리 대조할 수 있는 곳은 이 노드뿐이라 규정 위반을 conflicts에 따로 적게 합니다.
synthesizer_prompt = ChatPromptTemplate.from_messages([
    ("system", "여러 조사원의 메모를 하나의 출장 기획 메모로 합칩니다. "
     "메모끼리 맞춰 보고, 규정 상한을 넘는 선택지는 conflicts에 적으세요. 메모에 없는 내용과 계산값(예: 3박 합계)은 만들지 마세요."),
    ("human", "출장 요청: {request}\n\n조사 메모:\n{findings}"),
])

# 종합 체인: 프롬프트 -> LLM -> Brief 구조화 출력
synthesizer_chain = synthesizer_prompt | llm.with_structured_output(Brief)

In [ ]:
# [노드] 조사 종합
def synthesizer(state: TripState):
    """모든 조사 메모를 합쳐 안내문에 쓸 사실·주의할 점·조사하지 못한 항목으로 나눈 기획 메모를 반환합니다."""
    # 작업별 메모를 한 문자열로 묶어 넘기고, 받은 결과는 dict로 바꿔 State에 담습니다.
    brief = synthesizer_chain.invoke({
        "request": state["request"],
        "findings": format_findings(state["findings"]),
    })
    return {"brief": brief.model_dump()}

### 5-6. 그래프 연결과 실행

`planner`에서 `assign_research`로 `Send` 목록을 만들고, 모든 `researcher`가 끝나면 `synthesizer`가 한 번 실행됩니다. 실행하면 작업 수만큼 조사원 에이전트가 LLM을 호출합니다. API 사용량 한도가 걱정되면 `invoke(..., config={"max_concurrency": 2})`처럼 동시에 도는 실행 수를 제한할 수 있습니다. 작업 수는 그대로이고(예: 4개), 동시에 최대 2개까지만 실행할 뿐입니다.

In [ ]:
# 그래프 연결: planner -> researcher(작업 수만큼) -> synthesizer
builder = StateGraph(TripState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("synthesizer", synthesizer)

builder.add_edge(START, "planner")
# planner가 만든 작업 수만큼 researcher가 동시에 실행됩니다.
builder.add_conditional_edges("planner", assign_research, ["researcher"])
builder.add_edge("researcher", "synthesizer")
builder.add_edge("synthesizer", END)

research_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(research_app.get_graph().draw_mermaid_png()))

In [ ]:
# planner를 다시 호출하므로 위에서 본 작업 목록과 조금 다를 수 있습니다.
research_result = research_app.invoke(make_input(tokyo_request))
print("작업 수:", len(research_result["tasks"]), "조사 메모 수:", len(research_result["findings"]))

# synthesizer가 나눈 결과입니다. facts=안내문에 쓸 사실, conflicts=주의할 점, gaps=조사하지 못한 항목
print(research_result["brief"])

**확인할 점:** 숙소와 규정을 함께 조사했다면 `conflicts`(주의할 점)에 1박 상한(해외 20만 원)을 넘는 숙소가 적혀 있어야 합니다. 조사원 하나는 이 사실을 알 수 없고, 종합 단계에서 처음 드러납니다.

## 6. 핸드오프와 검수 루프

### 6-1. 핸드오프와 Command.PARENT

**핸드오프**는 일하던 에이전트가 스스로 다음 담당자를 골라 일을 넘기는 방식입니다. 에이전트 안의 도구가 `Command(goto=노드, graph=Command.PARENT)`를 반환하면 에이전트 실행은 그 자리에서 끝나고, 바깥 그래프(팀 그래프)의 `goto` 노드가 이어서 실행됩니다.

`PARENT`는 "부모 그래프"라는 뜻입니다. 교안 01에서 본 것처럼 `create_agent`로 만든 에이전트도 하나의 그래프이고, 팀 그래프의 노드 안에서 실행하면 팀 그래프가 부모, 에이전트가 자식(서브그래프)이 됩니다. 도구는 자식 그래프 안에서 실행되므로, `graph`를 지정하지 않으면 `goto`는 에이전트 그래프 안의 노드를 찾습니다. 팀 그래프의 `reviewer`로 가려면 `Command.PARENT`가 필요합니다.

<img src="images/11_handoff.png" width="1050" alt="팀 그래프(부모) 안의 writer 에이전트(자식)가 안내문을 다 쓰면 request_review로 reviewer에게, 공항에서 빅사이트까지 이동 정보가 없으면 request_research로 supervisor에게 넘깁니다. 두 도구 모두 Command.PARENT로 부모 그래프의 노드로 이동합니다.">

`writer` 에이전트에게 핸드오프 도구를 두 개 줍니다.

- `request_review`: 안내문을 다 쓰면 `update={"draft": 안내문}`과 함께 `reviewer`로 넘깁니다.
- `request_research`: 요청한 항목이 기획 메모에 없으면 무엇이 빠졌는지 적어 `supervisor`로 넘깁니다.

**예: 이동 정보가 빠진 기획 메모**

1. `writer`가 "도쿄 공항에서 빅사이트까지 가는 길이 메모에 없다"는 것을 알아챕니다.
2. `request_research`를 호출해 `supervisor`로 넘깁니다.
3. `supervisor`가 `planner`로 보내고, `planner`는 빠진 조사(현지 이동)만 다시 합니다.

아래는 6-7 팀 그래프에서 `writer` 주변만 그린 흐름입니다.

<img src="images/node_flow/l2_handoff_overview.png" width="1050" alt="synthesizer 다음 writer 에이전트가 실행됩니다. writer는 검수 요청이면 reviewer로, 추가 조사 요청이면 supervisor로 핸드오프합니다. reviewer 다음에는 supervisor가 실행됩니다.">

| | 교안 01의 `Command(goto)` | 핸드오프 |
|---|---|---|
| `Command`를 반환하는 곳 | 노드 함수 (예: `supervisor` 노드) | 에이전트 안의 도구 (예: `writer`의 `request_review`, `request_research`) |
| `graph` 인자 | 지정 안 함: 지금 그래프 안에서 이동 | `Command.PARENT`: 에이전트 바깥의 팀 그래프에서 이동 |
| 갈 곳을 정하는 쪽 | 슈퍼바이저 노드 | 일하던 에이전트 (예: `writer`가 `reviewer`와 `supervisor` 중 고름) |

**주의 두 가지**

- 핸드오프가 일어나면 `writer` 노드의 남은 코드와 반환값은 쓰이지 않습니다. 라운드 수(`round`)처럼 꼭 반영돼야 하는 값은 핸드오프가 없는 6-5의 `supervisor`가 갱신합니다.
- 바깥 State에는 `messages` 필드를 두지 않고, 안내문만 `draft`로 넘깁니다. writer의 대화 기록을 통째로 넘기면 다음 LLM 호출에서 오류가 나기 때문입니다.
  1. writer 모델이 `request_review(draft=안내문)`을 부르기로 합니다. 대화 기록 끝에는 이 도구 호출이 담긴 `AIMessage`가 쌓입니다.
  2. 도구는 `Command.PARENT`를 반환하는 순간 writer를 끝내고 `reviewer`로 넘어갑니다. 그래서 이 호출에 대한 도구 응답(`ToolMessage`)은 기록에 생기지 않습니다.
  3. 이 기록을 바깥으로 넘겨 다른 LLM에 그대로 주면, OpenAI API는 "도구 호출 뒤에 그 응답이 없다"는 오류를 냅니다.

### 6-2. 핸드오프 도구와 안내문 작성 노드 (`writer`)

`writer` 노드 안에서 에이전트를 실행합니다. 에이전트가 두 도구 중 하나를 부르면 그 순간 해당 노드로 넘어갑니다. 도구를 부르지 않고 끝나면 팀 그래프가 검수 없이 끝나지 않도록 마지막 답변을 안내문으로 `reviewer`에 넘깁니다.

**핸드오프 도구는 한 번에 하나만 부르게 합니다.** 모델은 한 응답에서 도구 여러 개를 함께 부를 수 있습니다. 예를 들어 writer가 안내문을 다 쓴 뒤 `request_review(draft=안내문)`과 `request_research(missing="이동 정보")`를 한 응답에서 함께 부르면 이렇게 됩니다.

1. 두 도구가 각각 `Command.PARENT`를 반환하지만, 팀 그래프는 그중 하나의 이동만 받아들입니다.
2. `request_research` 쪽이 받아들여지면 `supervisor`로 넘어갑니다. `reviewer`는 실행되지 않고, `request_review`에 담긴 안내문도 `draft`에 저장되지 않습니다.
3. 어느 쪽이 받아들여질지는 코드로 보장되지 않습니다.

그래서 `writer`의 모델에는 `parallel_tool_calls=False`를 주어 한 응답에 도구를 하나만 부르게 합니다. 이 설정은 모델 응답 하나에 도구 호출을 하나 이하로 제한할 뿐이고, `Send`로 노드를 동시에 실행하는 병렬과는 관계가 없습니다.

<img src="images/node_flow/l2_handoff_writer.png" width="1050" alt="writer: 안내문을 쓰면 request_review로 reviewer에게, 쓸 수 없으면 request_research로 supervisor에게 넘깁니다.">

In [ ]:
from langgraph.types import Command

# [핸드오프 도구] 검수 요청: 완성한 안내문을 reviewer에게 넘깁니다.
@tool
def request_review(draft: str) -> Command:
    """완성한 안내문을 검수자(reviewer)에게 넘깁니다. draft에 안내문 전문을 넣으세요."""
    # graph=Command.PARENT: 에이전트 그래프가 아니라 바깥 그래프의 reviewer로 이동합니다.
    # update는 바깥 State에 반영됩니다. 여기서는 draft 필드에 안내문을 저장합니다.
    return Command(goto="reviewer", graph=Command.PARENT, update={"draft": draft})


# [핸드오프 도구] 추가 조사 요청: 빠진 정보를 적어 supervisor에게 넘깁니다.
@tool
def request_research(missing: str) -> Command:
    """기획 메모만으로 안내문을 쓸 수 없을 때 무엇이 빠졌는지 알리고 추가 조사를 요청합니다."""
    # planner로 바로 가지 않고 supervisor를 거쳐야 라운드 상한(max_rounds) 검사를 받습니다.
    # reviewer 판정과 같은 모양으로 review 필드를 채워 supervisor가 같은 방식으로 읽게 합니다.
    review = {"passed": False, "issue": "research", "feedback": missing}
    return Command(goto="supervisor", graph=Command.PARENT, update={"review": review})

In [ ]:
# 두 도구를 언제 쓰는지 시스템 프롬프트에 적어 둡니다.
writer_system = ("사내 출장 지원 담당자로서 출장자에게 보낼 한국어 출장 안내문을 씁니다. "
                 "'출장 개요, 추천 항공편·숙소, 규정 위반 여부, 승인·예약 기한, 확인 필요' 순서로 짧게 쓰세요. "
                 "기획 메모의 사실만 쓰고 금액·시간은 그대로 옮기세요. 숙소는 1박 요금만 적고, 박수를 곱한 합계나 두 금액의 차액처럼 메모에 없는 계산값은 쓰지 마세요. "
                 "규정 상한을 넘는 선택지는 추천하지 마세요. "
                 "검수 의견이 있으면 반영하세요. 기획 메모의 facts는 안내문에 쓸 사실, conflicts는 주의할 점, gaps는 조사하지 못한 항목입니다. gaps의 항목은 '확인 필요'에 적고 넘어가세요. "
                 "쓰기 전에 출장 요청이 물은 항목을 하나씩 기획 메모와 대조하세요. 항목이 있다고 보는 기준은 "
                 "항공편=편명, 숙소=숙소 이름, 공항에서 행사장 가는 길=공항에서 출발하는 교통수단, 출장 규정=규정 조항입니다. "
                 "기준에 맞는 사실이 한 줄도 없는 항목이 있을 때만 "
                 "안내문을 쓰지 말고 request_research로 무엇이 빠졌는지 알리세요. 요금이나 다른 공항 경로처럼 세부만 빠졌으면 '확인 필요'에 적으세요. "
                 "다 쓰면 request_review 도구의 draft에 안내문 전문을 넣어 제출하세요.")

# 한 응답에 도구를 하나만 부르게 합니다. 핸드오프 두 개가 함께 나가면 한쪽만 반영됩니다.
writer_llm = ChatOpenAI(model=model_name, use_responses_api=True,
                        model_kwargs={"parallel_tool_calls": False})

# writer 에이전트: 핸드오프 도구 두 개만 가집니다.
writer_agent = create_agent(writer_llm, tools=[request_review, request_research],
                            system_prompt=writer_system)

In [ ]:
# [보조 함수] LLM 컨텍스트 만들기: writer 에이전트에 넣을 출장 요청·기획 메모(brief)·검수 의견을 한 문자열로 묶습니다.
def writer_request(state):
    """출장 요청, 기획 메모, 검수 의견을 writer에게 보낼 한 문자열로 묶습니다."""
    feedback = state["feedback"]
    # 첫 작성에는 검수 의견이 없으므로 "없음"을 넘깁니다.
    if not feedback:
        feedback = "없음"

    return (f"출장 요청: {state['request']}\n\n기획 메모:\n{state['brief']}\n\n"
            f"검수 의견: {feedback}")


# [노드] 안내문 작성: 다음 노드로 넘기는 일은 핸드오프 도구가 합니다.
def writer(state: TripState):
    """기획 메모와 검수 의견으로 writer 에이전트를 실행합니다."""
    result = writer_agent.invoke({"messages": [HumanMessage(writer_request(state))]})

    # 핸드오프가 일어나면 아래 줄은 실행되지 않습니다.
    # 도구를 부르지 않고 끝났을 때만 마지막 답변을 안내문으로 보고 reviewer에게 넘깁니다.
    return Command(goto="reviewer", update={"draft": result["messages"][-1].text})

### 6-3. 검수 노드 (`reviewer`)

`reviewer`는 안내문을 **원래 조사 메모**와 대조합니다. 반려할 때 문제 종류(`issue`)를 둘로 나눕니다. 글만 고치면 되면 `rewrite`, 조사 자체가 모자라면 `research`입니다. 6-5의 `supervisor`가 이 값으로 다음 단계를 고릅니다.

<img src="images/node_flow/l2_handoff_reviewer.png" width="1050" alt="reviewer: 핸드오프로 받은 안내문을 조사 메모와 대조해 판정하고, 판정은 supervisor가 읽습니다.">

In [ ]:
# 검수 판정 스키마: 통과 여부·문제 종류·의견
class Review(BaseModel):
    passed: bool = Field(description="고치지 않고 출장자에게 보내도 되면 true")
    issue: Literal["none", "rewrite", "research"] = Field(
        description="통과면 none, 글 수정이면 rewrite, 조사 보완이면 research"
    )
    feedback: str = Field(
        description="반려 이유와 할 일 한두 문장. rewrite이면 안내문에서 고칠 문장을 따옴표로 인용하고, "
                    "research이면 다시 조사할 항목(예: 공항에서 빅사이트까지 이동 방법)을 적는다. 통과(none)면 빈 문자열"
    )

# 검수 프롬프트: 안내문을 조사 메모와 대조해 문제 종류를 고르게 합니다.
reviewer_prompt = ChatPromptTemplate.from_messages([
    ("system", "출장 안내문을 조사 메모와 대조해 검수합니다. "
     "안내문의 금액·시간을 하나씩 조사 메모에서 찾아 보세요. 메모에 그대로 없는 값(1박 요금에 박수를 곱한 합계, 두 금액의 차액 포함)이 하나라도 있거나 "
     "규정 상한을 넘는 선택지를 추천했으면 issue=rewrite입니다. "
     "요청한 출장에 해당하는 승인·예약 기한이 조사 메모에 있는데 안내문에 없거나 다르게 적혀 있으면 issue=rewrite입니다. "
     "요청한 항목의 조사 메모 자체가 없어 안내문을 완성할 수 없으면 issue=research입니다. 세부만 빠지고 '확인 필요'에 적혀 있으면 문제로 보지 마세요. "
     "문제가 없으면 passed=true, issue=none입니다."),
    ("human", "출장 요청: {request}\n\n조사 메모:\n{findings}\n\n안내문:\n{draft}"),
])

# 검수 체인: 프롬프트 -> LLM -> Review 구조화 출력
reviewer_chain = reviewer_prompt | llm.with_structured_output(Review)

In [ ]:
# [노드] 검수
def reviewer(state: TripState):
    """안내문을 원래 조사 메모와 대조한 검수 결과를 반환합니다."""
    # 기획 메모가 아니라 원래 조사 메모와 대조해야 synthesizer가 옮겨 적다 생긴 실수까지 잡습니다.
    review = reviewer_chain.invoke({
        "request": state["request"],
        "findings": format_findings(state["findings"]),
        "draft": state["draft"],
    })

    return {"review": review.model_dump()}

### 6-4. 검수 결과에 따라 다시 돌기

**검수 루프**는 `reviewer`가 반려하면 앞 단계로 돌려보내 다시 하게 하는 반복입니다. **라운드 가드**는 그 반복 횟수의 상한입니다.

상한 2일 때:

1. `reviewer`가 "규정 상한을 넘는 숙소를 추천함"으로 반려 → `supervisor`가 `round`를 0에서 1로 올리고 `writer`로 보냅니다.
2. 다시 반려 → `round` 1에서 2, 다시 `writer`로 보냅니다.
3. 또 반려 → `round` 2 = 상한이므로 더 돌지 않고 "라운드 상한 도달: 사람 확인 필요"로 끝냅니다. 마지막 검수 의견은 `review`에 남습니다.

<img src="images/12_review_loop.png" width="1050" alt="상한 2일 때 첫 반려(상한 넘는 숙소 추천)로 round 0에서 1, 다시 반려로 1에서 2, 세 번째 반려에서 round 2가 상한이라 사람 확인 필요로 마칩니다. 통과하면 바로 완료합니다.">

`writer`가 `request_research`로 넘긴 요청도 `research` 판정으로 `supervisor`에 들어오므로 같은 라운드 가드를 받습니다.

<img src="images/node_flow/l2_team_overview.png" width="1050" alt="planner, researcher, synthesizer, writer를 거쳐 핸드오프로 reviewer에 갑니다. supervisor는 다시 쓰기면 writer, 다시 조사면 planner, 통과하거나 라운드 상한에 닿으면 END로 보냅니다.">

| `reviewer` 판정 (`review`) | 지금 `round` | `supervisor`가 보내는 곳 |
|---|---|---|
| `passed=True` (예: 메모에 없는 금액도 규정 위반도 없음) | 상관없음 | `END`, 상태 "통과" (바로 완료) |
| `issue="rewrite"` (예: 상한 넘는 숙소를 추천함) | 상한 미만 (0 또는 1) | `writer` (검수 의견을 `feedback`에 담아) |
| `issue="research"` (예: 숙소 조사 메모가 없음) | 상한 미만 (0 또는 1) | `planner` (빠진 조사만 다시 계획) |
| 통과 못 함 | 상한 도달 (2) | `END`, 상태 "라운드 상한 도달: 사람 확인 필요" |

### 6-5. 다음 단계 결정 노드 (`supervisor`)

LLM 없이 `review`(판정)와 `round`·`max_rounds`만 읽습니다. 이동은 교안 01처럼 `Command(goto)`로 반환합니다.

<img src="images/node_flow/l2_team_supervisor.png" width="1050" alt="supervisor: reviewer 판정과 라운드 수로 다시 쓰기, 다시 조사, 끝 중 하나를 고릅니다.">

In [ ]:
# [노드] 다음 단계 결정
def supervisor(state: TripState) -> Command[Literal["planner", "writer", "__end__"]]:
    """검수 판정과 라운드 수로 끝낼지, 다시 쓸지, 다시 조사할지 정합니다."""
    review = state["review"]

    # 통과면 라운드와 관계없이 끝냅니다.
    if review["passed"]:
        return Command(goto=END, update={"status": "통과"})

    # 라운드 가드: 상한에 닿으면 더 돌리지 않고 사람에게 넘깁니다.
    if state["round"] >= state["max_rounds"]:
        return Command(goto=END, update={"status": "라운드 상한 도달: 사람 확인 필요"})

    # 조사가 모자라면 planner로, 글만 고치면 되면 writer로 돌려보냅니다.
    if review["issue"] == "research":
        next_node = "planner"
    else:
        next_node = "writer"

    # 라운드를 1 올리고 검수 의견을 feedback에 담습니다. 핸드오프로 끊기지 않는 노드라 항상 반영됩니다.
    return Command(goto=next_node,
                   update={"round": state["round"] + 1, "feedback": review["feedback"]})

**실행 확인:** 판정 세 가지를 직접 넣어 봅니다. `END`는 문자열 `"__end__"`입니다.

In [ ]:
# 검수 결과 세 가지를 직접 넣어 supervisor가 고르는 길을 봅니다.
passed = {
    "review": {"passed": True, "issue": "none", "feedback": ""},
    "round": 0,
    "max_rounds": 2,
}
need_research = {
    "review": {"passed": False, "issue": "research", "feedback": "숙소 조사가 없습니다."},
    "round": 0,
    "max_rounds": 2,
}
at_limit = {
    "review": {"passed": False, "issue": "rewrite", "feedback": "요금이 메모와 다릅니다."},
    "round": 2,
    "max_rounds": 2,
}

# 통과 판정은 라운드와 관계없이 끝으로 갑니다.
print("통과:", supervisor(passed).goto)

# 조사가 모자라고 라운드가 남았으면 planner로 돌려보냅니다.
print("조사 부족, 라운드 0:", supervisor(need_research).goto)

# 라운드가 상한에 닿으면 판정과 관계없이 끝냅니다.
print("상한 도달, 라운드 2:", supervisor(at_limit).goto)

### 6-6. 반려되는 안내문으로 루프 확인

LLM 판정은 실행마다 달라서 팀 실행에서 반려가 안 나올 수도 있습니다. 그래서 먼저 **사람이 일부러 틀리게 쓴 안내문**을 실제 `reviewer`에 넣어 반려와 그다음 경로를 확인합니다. 5절 조사 메모(`research_result["findings"]`)를 기준으로 판정합니다. 승인 기한 누락은 조사 메모에 승인 규정이 있을 때만 잡을 수 있고, planner가 승인 규정을 조사했는지는 실행마다 다릅니다.

이 안내문에는 실무에서 자주 나오는 실수 세 가지를 넣었습니다.

- 해외 숙박 상한(1박 200,000원)을 넘는 시나가와 프라임(1박 210,000원)을 추천
- 조사 메모와 다른 항공 요금(KE2101을 350,000원으로 적음)
- 해외 출장 승인 기한(출발 7일 전) 누락

In [ ]:
# 사람이 일부러 틀리게 쓴 안내문입니다. 모델이 만든 것이 아니라 노트북에 직접 적은 글입니다.
flawed_draft = """도쿄 빅사이트 박람회 출장 안내 (10월 21일~24일, 3박 4일)

추천 항공편: KE2101 김포 08:20 출발, 하네다 10:30 도착, 350,000원
추천 숙소: 시나가와 프라임 1박 210,000원 (도쿄 빅사이트 전철 25분)
현지 이동: 하네다공항에서 리무진버스로 약 30분"""

# 시작 상태에 조사 메모와 틀린 안내문을 넣고 reviewer를 직접 호출합니다.
flawed_state = make_input(tokyo_request)
flawed_state["findings"] = research_result["findings"]
flawed_state["draft"] = flawed_draft

flawed_state.update(reviewer(flawed_state))

flawed_review = flawed_state["review"]
print("통과 여부:", flawed_review["passed"], "| 문제 종류:", flawed_review["issue"])

# reviewer가 적은 반려 이유입니다. 다시 쓸 때 writer에게 이 의견이 전달됩니다.
print("검수 의견:", flawed_review["feedback"])

반려된 판정을 `supervisor`에 넘기면 다음 노드와 바뀌는 값이 정해집니다. 지금 `round`는 0이고 상한은 2입니다.

In [ ]:
# supervisor가 돌려주는 Command의 goto는 다음 노드, update는 State에 반영할 값입니다.
route = supervisor(flawed_state)
print("다음 노드:", route.goto)

# update에는 1 오른 round와 다음 노드에 전할 검수 의견(feedback)이 담깁니다.
print("바뀌는 값:", route.update)

### 6-7. 팀 그래프 연결

5절과 6-2·6-3의 노드를 그대로 쓰고, `reviewer` 뒤에 `supervisor`를 붙입니다. `supervisor`의 반환 타입 `Command[Literal[...]]`는 그래프가 supervisor의 다음 노드 후보(planner, writer, END)를 알게 해 줍니다.

In [ ]:
# 조사·종합 노드와 writer·reviewer 노드를 잇고, reviewer 뒤에 supervisor를 붙입니다.
builder = StateGraph(TripState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("synthesizer", synthesizer)
# writer가 핸드오프로 갈 수 있는 곳(reviewer, supervisor)을 그래프 그림에 표시합니다. 실행에는 영향이 없습니다.
builder.add_node("writer", writer, destinations=("reviewer", "supervisor"))
builder.add_node("reviewer", reviewer)
builder.add_node("supervisor", supervisor)

builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", assign_research, ["researcher"])
builder.add_edge("researcher", "synthesizer")
builder.add_edge("synthesizer", "writer")
builder.add_edge("reviewer", "supervisor")

# supervisor에서 나가는 엣지는 만들지 않습니다. 반환하는 Command가 다음 노드를 정합니다.
team_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(team_app.get_graph().draw_mermaid_png()))

### 6-8. 팀 실행과 결과 저장

도쿄 박람회 요청으로 처음부터 끝까지 실행합니다. 라운드 상한은 2입니다. 첫 안내문이 통과하면 `supervisor`를 한 번 거치고 바로 끝나고, 반려되면 6-6처럼 `writer`나 `planner`로 돌아갑니다.

In [ ]:
# 노드가 끝날 때마다 이름을 모아 방문 순서를 기록합니다.
visited = []
for mode, chunk in team_app.stream(make_input(tokyo_request, max_rounds=2),
                                   stream_mode=["updates", "values"]):
    if mode == "updates":
        # updates는 {노드 이름: 그 노드가 바꾼 값} 모양입니다.
        for node_name in chunk:
            visited.append(node_name)
    else:
        # values는 노드가 끝날 때마다 받는 전체 State라 마지막으로 받은 값이 최종 결과입니다.
        final = chunk

print("방문 순서:", " -> ".join(visited))

# round는 supervisor가 돌려보낸 횟수, status는 끝난 이유입니다.
print("돌려보낸 횟수:", final["round"], "| 처리 상태:", final["status"])

# 마지막 판정입니다. reviewer 또는 request_research가 review 필드에 쓴 값입니다.
last_review = final["review"]
print("통과 여부:", last_review["passed"], "| 문제 종류:", last_review["issue"], "| 검수 의견:", last_review["feedback"])

# supervisor가 writer에게 돌려보낼 때 전한 마지막 의견입니다. planner로 보낸 의견은 planner가 반영한 뒤 비웁니다.
if final["feedback"]:
    print("돌려보낸 의견:", final["feedback"])
else:
    print("돌려보낸 의견: 없음")

처리 상태가 "통과"일 때만 출장자에게 보낼 안내문입니다. 라운드 상한에 닿아 끝났다면 아래 글은 마지막으로 반려된 글이므로, 위의 검수 의견을 보고 사람이 고친 뒤 보냅니다(추가 조사 요청만 하다 끝났다면 비어 있을 수 있습니다).

In [ ]:
# 마지막으로 저장된 안내문(draft)을 마크다운으로 봅니다.
display(Markdown(final["draft"]))

In [ ]:
# 안내문과 조사 메모를 output 폴더에 남깁니다. 반려된 글과 구분되도록 첫 줄에 처리 상태를 적습니다.
(output_dir / "lesson02_trip_guide.md").write_text(
    f"처리 상태: {final['status']}\n\n{final['draft']}", encoding="utf-8"
)
(output_dir / "lesson02_findings.json").write_text(
    json.dumps(final["findings"], ensure_ascii=False, indent=2), encoding="utf-8"
)

print("저장:", output_dir / "lesson02_trip_guide.md", output_dir / "lesson02_findings.json")

**확인할 점:** 방문 순서에서 `researcher`가 작업 수만큼 나오는지, `writer` 다음에 `reviewer`가 오는지, `supervisor` 뒤에 어디로 갔는지 읽습니다. 다시 조사했다면 `findings`에 새 작업 ID가 추가되거나 같은 ID의 메모가 바뀌었는지 JSON에서 확인합니다.

## 7. 핵심 정리

- **Send:** 라우팅 함수가 `Send` 목록을 돌려주면 그 수만큼 같은 노드가 동시에 실행됩니다(예: 도쿄 요청의 작업 4개 → `researcher` 4번). 각 실행은 담아 보낸 값만 받습니다(예: `{"task": ...}`).
- **리듀서:** 필드에 들어온 새 값을 기존 값과 합치는 규칙입니다. 같은 단계에 결과가 동시에 오는 필드에는 꼭 필요합니다. 없으면 에러, `operator.add`는 이어 붙이기, dict 병합은 작업 ID별 교체입니다(예: `hotel_tokyo`를 다시 조사하면 그 메모만 바뀜). 리듀서 필드에는 기존 값 없이 새 결과만 반환합니다.
- **종합:** 따로 조사한 메모를 맞춰 봅니다(예: 숙소 1박 210,000원 + 규정 상한 200,000원 → 상한 초과 주의).
- **핸드오프:** 에이전트 안의 도구가 `Command.PARENT`로 바깥 그래프의 다음 노드를 부릅니다(예: `writer`가 메모가 충분하면 `request_review` → `reviewer`, 빠진 항목이 있으면 `request_research` → `supervisor`). 조건부 엣지와 달리 갈 곳을 일하던 에이전트가 고릅니다. `PARENT`는 에이전트(자식 그래프)를 실행한 팀 그래프(부모)를 가리키며, 핸드오프 도구가 함께 불리지 않도록 `parallel_tool_calls=False`를 줍니다.
- **검수 루프와 라운드 가드:** `reviewer`가 반려하면 `supervisor`가 `round`를 올리며 `writer`나 `planner`로 돌려보내고, 상한(예: 2)에 닿으면 "사람 확인 필요"로 마칩니다. 횟수는 핸드오프로 중단되지 않는 `supervisor`가 셉니다.